In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install menpo
!git clone https://github.com/im-xiaoming/HuyenLaoNhao.git

In [ ]:
!cp -r /content/drive/MyDrive/Ying/faces_extracted.zip /content
!unzip -q /content/faces_extracted.zip -d /content/

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from xiaoying.utils import get_loader, get_model, get_head, get_optimizer
from xiaoying.trainer import Trainer

In [ ]:
TRANSFORMS = transforms.Compose([
        transforms.Resize((112, 112)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])


DATA_PATH = '/content/faces_extracted'
CHECKPOINT = ''
BATCH_SIZE = 512
MODEL_NAME = 'ir_18'
LEARNING_RATE = 0.1
EPOCHS = 5

# Head params
m = 0.4
h = 0.333
s = 64
t_alpha = 0.99
CLASS_NUM=8631
OUTPUT_SIZE = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
criterion = nn.CrossEntropyLoss()

dataloader = get_loader(data_path=DATA_PATH, transforms=TRANSFORMS, batch_size=BATCH_SIZE, shuffle=True)
model = get_model(MODEL_NAME, device)
head = get_head(device, OUTPUT_SIZE, CLASS_NUM, m, h, s, t_alpha)
optimizer = get_optimizer(model=model, head=head, lr=LEARNING_RATE)

In [ ]:
trainer = Trainer(dataloader, model, head, optimizer, criterion, device, EPOCHS)
trainer._load_checkpoint(CHECKPOINT)
trainer.train()